# Study Case 01 — Complex SQL Analysis

This notebook documents advanced SQL analysis. SQL performs the calculations; Python displays the returned result sets.

Topics: funding dependency, revenue concentration, financial sustainability, workforce capacity, volunteer intensity, segment benchmarking and outlier analysis.


In [ ]:
import os, psycopg
from IPython.display import display
conn = psycopg.connect(os.environ['SUPABASE_DB_URL'])


In [ ]:
sql = """
WITH ranked AS (
  SELECT charity_id, total_revenue, revenue_from_government,
         SUM(total_revenue) OVER () AS total_sector_revenue,
         SUM(total_revenue) OVER (ORDER BY total_revenue DESC ROWS UNBOUNDED PRECEDING) AS cumulative_revenue
  FROM acnc.financials
  WHERE total_revenue > 0
)
SELECT *, cumulative_revenue / NULLIF(total_sector_revenue,0) AS cumulative_share
FROM ranked
ORDER BY total_revenue DESC
LIMIT 100;
"""
with conn.cursor() as cur:
    cur.execute(sql)
    display(cur.fetchall())


In [ ]:
sql = """
SELECT
  charity_size,
  COUNT(*) AS charities,
  SUM(total_revenue) AS revenue,
  percentile_cont(0.5) WITHIN GROUP (ORDER BY total_revenue) AS median_revenue,
  percentile_cont(0.5) WITHIN GROUP (ORDER BY net_surplus_deficit) AS median_net_surplus,
  percentile_cont(0.5) WITHIN GROUP (ORDER BY total_full_time_equivalent_staff) AS median_fte,
  percentile_cont(0.5) WITHIN GROUP (ORDER BY staff_volunteers) AS median_volunteers
FROM analytics.v_acnc_charity_summary
GROUP BY charity_size
ORDER BY revenue DESC;
"""
with conn.cursor() as cur:
    cur.execute(sql)
    display(cur.fetchall())
